# TrustCart — CNN Product Recognition (RPC) · v2 anti-overfitting

**Dataset**: `diyer22/retail-product-checkout-dataset` · **Task**: classify checkout product crops into
**17 retail supercategories** with **MobileNetV2** transfer learning.

v1 overfitted (train acc ≈ 0.99, val ≈ 0.69, test 0.59). Main cause: only 3 000 checkout crops for a
turntable → checkout domain gap. v2 fixes:

| Problem (v1) | Fix (v2) |
|---|---|
| 3 000 checkout crops | **all** crops of 80 % of `val2019` images (~59 k) + 8 000 class-balanced turntable crops |
| flip + light colour jitter | rotations 90°, both flips, random zoom-crop, hue/saturation/brightness/contrast |
| plain cross-entropy, Adam | **label smoothing 0.1**, **AdamW weight decay**, dropout 0.4, class weights |
| early stop on `val_accuracy`, patience 2 | early stop on **`val_loss`**, patience 3, cosine LR in fine-tuning |
| 1 500 test crops, no train metric | ~10 k test crops (whole `test2019` images) + clean train / val / test metrics and gap |
| over-confident softmax | **temperature scaling** fitted on val, baked into the exported model |

Leakage rule: every split is grouped by **image** (no crop of a test/val image is ever trained on);
`test2019` is never used for training or model selection.

In [ ]:
# Phase 0: Environment & configuration
import json
import os
import random
import shutil
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SMOKE = bool(os.environ.get('TRUSTCART_SMOKE'))   # tiny local dry run on a fake dataset
DATA_DIR = Path(os.environ.get('TRUSTCART_RPC_DIR', '/kaggle/input/retail-product-checkout-dataset'))
if not DATA_DIR.exists():
    DATA_DIR = Path('/kaggle/input/datasets/diyer22/retail-product-checkout-dataset')
WORK_DIR = Path(os.environ.get('TRUSTCART_WORK_DIR', '/kaggle/working'))
OUT_DIR = WORK_DIR / 'cnn_export'
CROP_DIR = WORK_DIR / 'crops'
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16 if SMOKE else 64
N_TURNTABLE = 40 if SMOKE else 8000          # class-balanced turntable crops
N_TEST_IMAGES = 4 if SMOKE else 800          # whole test2019 images (~12 crops each)
EPOCHS_HEAD = 1 if SMOKE else 6
EPOCHS_FINE = 1 if SMOKE else 14
FINE_TUNE_LAYERS = 50
LABEL_SMOOTHING = 0.1
WEIGHTS = None if SMOKE else 'imagenet'

print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'), '| DATA_DIR:', DATA_DIR)

In [ ]:
# Phase 1: COCO annotations -> crop records with 17 supercategory labels
def load_split(split_name):
    with open(DATA_DIR / f'instances_{split_name}2019.json') as f:
        data = json.load(f)
    return data, {img['id']: img for img in data['images']}, {c['id']: c for c in data['categories']}

train_data, train_imgs, train_cats = load_split('train')
val_data, val_imgs, val_cats = load_split('val')
test_data, test_imgs, test_cats = load_split('test')

supercategories = sorted({c['supercategory'] for c in train_data['categories']})
super_to_idx = {s: i for i, s in enumerate(supercategories)}
NUM_CLASSES = len(supercategories)
print(f'Classes ({NUM_CLASSES}):', supercategories)

def build_records(data, img_map, cat_map, img_subdir):
    records = []
    for ann in data['annotations']:
        img, cat = img_map[ann['image_id']], cat_map[ann['category_id']]
        x, y, w, h = ann['bbox']
        if w < 10 or h < 10:
            continue
        records.append({'image_id': ann['image_id'], 'path': str(DATA_DIR / img_subdir / img['file_name']),
                        'bbox': (float(x), float(y), float(w), float(h)),
                        'label': super_to_idx[cat['supercategory']], 'supercategory': cat['supercategory']})
    return records

turntable_all = build_records(train_data, train_imgs, train_cats, 'train2019')
checkout_val_all = build_records(val_data, val_imgs, val_cats, 'val2019')
checkout_test_all = build_records(test_data, test_imgs, test_cats, 'test2019')
print(f'Raw crops -> turntable: {len(turntable_all)}, val2019: {len(checkout_val_all)}, test2019: {len(checkout_test_all)}')

In [ ]:
# Phase 2: Image-grouped, leakage-free splits
rng = random.Random(SEED)

# val2019 checkout images: 80 % train / 20 % validation, split by IMAGE
val_image_ids = sorted({r['image_id'] for r in checkout_val_all})
tr_ids, ev_ids = train_test_split(val_image_ids, test_size=0.20, random_state=SEED)
tr_ids, ev_ids = set(tr_ids), set(ev_ids)
assert not (tr_ids & ev_ids), 'image leakage between train and val'
checkout_train = [r for r in checkout_val_all if r['image_id'] in tr_ids]
val_records = [r for r in checkout_val_all if r['image_id'] in ev_ids]

# turntable crops: same count per class, so rare classes are not drowned out
by_class = {}
for r in turntable_all:
    by_class.setdefault(r['label'], []).append(r)
per_class = max(1, N_TURNTABLE // NUM_CLASSES)
turntable = [r for recs in by_class.values() for r in rng.sample(recs, min(per_class, len(recs)))]

train_records = checkout_train + turntable
rng.shuffle(train_records)

# test2019: all crops of N whole images, never seen in training or model selection
test_image_ids = sorted({r['image_id'] for r in checkout_test_all})
test_pick = set(rng.sample(test_image_ids, min(N_TEST_IMAGES, len(test_image_ids))))
test_records = [r for r in checkout_test_all if r['image_id'] in test_pick]

print(f'train: {len(train_records)} (checkout={len(checkout_train)}, turntable={len(turntable)})'
      f' | val: {len(val_records)} | test: {len(test_records)} crops from {len(test_pick)} images')
print('train class counts:', sorted(Counter(r['supercategory'] for r in train_records).items()))

In [ ]:
# Phase 3: Materialise 224x224 crops once (each source photo opened once, in parallel threads)
def materialize_crops(records, split_dir):
    out_dir = CROP_DIR / split_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    by_img = {}
    for idx, r in enumerate(records):
        by_img.setdefault(r['path'], []).append((idx, r))
    paths = [None] * len(records)

    def work(item):
        img_path, items = item
        with Image.open(img_path) as im:
            im = im.convert('RGB')
            W, H = im.size
            for idx, r in items:
                x, y, w, h = r['bbox']
                pad_x, pad_y = 0.08 * w, 0.08 * h
                box = (max(0, int(x - pad_x)), max(0, int(y - pad_y)),
                       min(W, int(x + w + pad_x)), min(H, int(y + h + pad_y)))
                out_p = out_dir / f'{idx:06d}.jpg'
                im.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR).save(out_p, quality=90)
                paths[idx] = str(out_p)

    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
        list(pool.map(work, by_img.items()))
    assert all(paths), 'some crops were not written'
    return paths, [r['label'] for r in records]

tr_paths, tr_labels = materialize_crops(train_records, 'train')
va_paths, va_labels = materialize_crops(val_records, 'val')
te_paths, te_labels = materialize_crops(test_records, 'test')
print('Crops materialised:', len(tr_paths), len(va_paths), len(te_paths))

In [ ]:
# Phase 4: tf.data with strong, checkout-realistic augmentation (top-down camera -> any rotation)
AUTOTUNE = tf.data.AUTOTUNE

def decode(path):
    return tf.cast(tf.image.decode_jpeg(tf.io.read_file(path), channels=3), tf.float32)

def augment(img):
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    scale = tf.random.uniform([], 0.75, 1.0)
    side = tf.cast(scale * IMG_SIZE, tf.int32)
    img = tf.image.random_crop(img, [side, side, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.image.random_brightness(img, 25.0)
    img = tf.image.random_contrast(img, 0.75, 1.25)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_hue(img, 0.04)
    return tf.clip_by_value(img, 0.0, 255.0)

def make_ds(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    def load(path, label):
        img = decode(path)
        if training:
            img = augment(img)
        return img, tf.one_hot(label, NUM_CLASSES)
    return ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(tr_paths, tr_labels, training=True)
val_ds = make_ds(va_paths, va_labels)
test_ds = make_ds(te_paths, te_labels)

# clean (non-augmented) train subset, to measure the real train/test gap
clean_idx = np.random.default_rng(SEED).choice(len(tr_paths), size=min(5000, len(tr_paths)), replace=False)
train_eval_paths = [tr_paths[i] for i in clean_idx]
train_eval_labels = [tr_labels[i] for i in clean_idx]
train_eval_ds = make_ds(train_eval_paths, train_eval_labels)

# sqrt-inverse class weights (milder than fully balanced, keeps calibration sane)
counts = np.bincount(tr_labels, minlength=NUM_CLASSES).astype(float)
class_weight = {i: float(np.sqrt(counts.sum() / (NUM_CLASSES * max(c, 1.0)))) for i, c in enumerate(counts)}
print('class weights:', {supercategories[i]: round(w, 2) for i, w in class_weight.items()})

In [ ]:
# Phase 5: MobileNetV2 -> logits; frozen head stage, then regularised fine-tuning
base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                         weights=WEIGHTS)
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)              # BatchNorm stays in inference mode, also while fine-tuning
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
logits = tf.keras.layers.Dense(NUM_CLASSES, kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                               name='logits')(x)
model = tf.keras.Model(inputs, logits, name='trustcart_cnn_mobilenetv2')

loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTHING)
metrics = [tf.keras.metrics.CategoricalAccuracy(name='accuracy')]

def callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

print('--- Stage 1: classification head (backbone frozen) ---')
model.compile(optimizer=tf.keras.optimizers.AdamW(1e-3, weight_decay=1e-4), loss=loss, metrics=metrics)
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight,
                  callbacks=callbacks(), verbose=2)

print(f'--- Stage 2: fine-tune top {FINE_TUNE_LAYERS} backbone layers ---')
base.trainable = True
for layer in base.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
steps = max(1, EPOCHS_FINE * int(np.ceil(len(tr_paths) / BATCH_SIZE)))
schedule = tf.keras.optimizers.schedules.CosineDecay(3e-5, decay_steps=steps, alpha=0.05)
model.compile(optimizer=tf.keras.optimizers.AdamW(schedule, weight_decay=1e-4), loss=loss, metrics=metrics)
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, class_weight=class_weight,
                  callbacks=callbacks(), verbose=2)

In [ ]:
# Phase 6: Learning curves (saved for the report)
history = {k: hist1.history[k] + hist2.history.get(k, []) for k in hist1.history}
switch = len(hist1.history['loss'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, key, title in ((axes[0], 'accuracy', 'Accuracy'), (axes[1], 'loss', 'Loss')):
    ax.plot(history[key], label=f'Train {key} (augmenté)')
    ax.plot(history[f'val_{key}'], label=f'Val {key}')
    ax.axvline(switch - 0.5, color='grey', ls='--', label='Début fine-tuning')
    ax.set_title(f'Évolution — {title}'); ax.set_xlabel('Époque'); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'courbes_apprentissage_cnn.png', dpi=150)
with open(OUT_DIR / 'cnn_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.items()} | {'fine_tune_start_epoch': switch}, f, indent=1)

In [ ]:
# Phase 7: Temperature scaling on val (fixes over-confidence; does not change predictions)
def logits_of(ds):
    return model.predict(ds, verbose=0).astype(np.float64)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(z, y, T):
    return float(-np.mean(np.log(softmax(z / T)[np.arange(len(y)), y] + 1e-12)))

val_logits, y_val = logits_of(val_ds), np.array(va_labels)
temps = np.linspace(0.25, 4.0, 376)
TEMPERATURE = float(temps[np.argmin([nll(val_logits, y_val, t) for t in temps])])
assert temps[0] < TEMPERATURE < temps[-1] or SMOKE, 'temperature hit the search bound'
print(f'Temperature: {TEMPERATURE:.3f}  (val NLL {nll(val_logits, y_val, 1.0):.4f} -> {nll(val_logits, y_val, TEMPERATURE):.4f})')

In [ ]:
# Phase 8: Clean train / val / test metrics, overfitting gap, calibration, confusion matrix
def ece_score(y, probs, n_bins=15):
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    bins = np.linspace(0, 1, n_bins + 1)
    return float(sum(np.mean(m) * abs(np.mean(pred[m] == y[m]) - np.mean(conf[m]))
                     for lo, hi in zip(bins[:-1], bins[1:])
                     if np.any(m := (conf > lo) & (conf <= hi))))

def evaluate(z, y):
    p_raw, p_cal = softmax(z), softmax(z / TEMPERATURE)
    pred = p_cal.argmax(axis=1)
    return {'accuracy': round(float(accuracy_score(y, pred)), 4),
            'macro_f1': round(float(f1_score(y, pred, average='macro')), 4),
            'ece_uncalibrated': round(ece_score(y, p_raw), 4),
            'ece': round(ece_score(y, p_cal), 4),
            'n_crops': int(len(y))}

y_tr_eval, y_test = np.array(train_eval_labels), np.array(te_labels)
test_logits = logits_of(test_ds)
results = {'train': evaluate(logits_of(train_eval_ds), y_tr_eval),
           'val': evaluate(val_logits, y_val),
           'test': evaluate(test_logits, y_test)}
gap = round(results['train']['accuracy'] - results['test']['accuracy'], 4)
for split, m in results.items():
    print(f'{split:5s}', m)
print(f'Overfitting gap (train - test accuracy): {gap:+.4f}')

test_pred = softmax(test_logits / TEMPERATURE).argmax(axis=1)
report = classification_report(y_test, test_pred, labels=range(NUM_CLASSES), target_names=supercategories,
                               digits=3, zero_division=0, output_dict=True)
print(classification_report(y_test, test_pred, labels=range(NUM_CLASSES), target_names=supercategories,
                            digits=3, zero_division=0))

cm = confusion_matrix(y_test, test_pred, labels=range(NUM_CLASSES), normalize='true')
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES), supercategories, rotation=90); ax.set_yticks(range(NUM_CLASSES), supercategories)
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel'); ax.set_title('Matrice de confusion normalisée (test2019)')
fig.colorbar(im); plt.tight_layout(); plt.savefig(OUT_DIR / 'matrice_confusion_cnn.png', dpi=150)

In [ ]:
# Phase 9: Export — calibrated softmax baked in, so the app keeps feeding raw [0,255] RGB and reading probabilities
export_in = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
z = model(export_in)
z = tf.keras.layers.Rescaling(1.0 / TEMPERATURE, name='temperature_scaling')(z)
export_model = tf.keras.Model(export_in, tf.keras.layers.Softmax(name='probabilities')(z),
                              name='trustcart_cnn_mobilenetv2_calibrated')
export_model.save(OUT_DIR / 'cnn_model.keras')

reloaded = tf.keras.models.load_model(OUT_DIR / 'cnn_model.keras', compile=False)
check = np.asarray(reloaded.predict(test_ds.take(1), verbose=0))
assert np.allclose(check.sum(axis=1), 1.0, atol=1e-4), 'exported model must output probabilities'
assert np.allclose(check, softmax(test_logits[:len(check)] / TEMPERATURE), atol=1e-3), 'export mismatch'

meta = {
    'version': 2,
    'backbone': 'MobileNetV2',
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'supercategories': supercategories,
    'input': 'raw RGB in [0, 255]; preprocessing, temperature scaling and softmax are inside the model',
    'temperature': round(TEMPERATURE, 4),
    'test_metrics': {'accuracy': results['test']['accuracy'], 'macro_f1': results['test']['macro_f1'],
                     'ece': results['test']['ece'], 'n_test_crops': results['test']['n_crops']},
    'train_metrics_clean': results['train'],
    'val_metrics': results['val'],
    'test_metrics_full': results['test'],
    'overfitting_gap_train_minus_test_accuracy': gap,
    'per_class_test_f1': {c: round(report[c]['f1-score'], 3) for c in supercategories},
    'splits': {'train_crops': len(tr_paths), 'train_checkout_crops': len(checkout_train),
               'train_turntable_crops': len(turntable), 'val_crops': len(va_paths),
               'test_crops': len(te_paths), 'test_images': len(test_pick),
               'grouping': 'by image; val = 20 % of val2019 images; test = whole test2019 images, never trained on'},
    'training': {'label_smoothing': LABEL_SMOOTHING, 'optimizer': 'AdamW (wd 1e-4)', 'dropout': 0.4,
                 'head_lr': 1e-3, 'fine_tune_lr': '3e-5 cosine', 'fine_tune_layers': FINE_TUNE_LAYERS,
                 'early_stopping': 'val_loss, patience 3, restore best', 'class_weight': 'sqrt-inverse frequency',
                 'epochs_run': len(history['loss']), 'seed': SEED,
                 'augmentation': 'rot90, h/v flip, zoom-crop 0.75-1, brightness, contrast, saturation, hue'},
    'tensorflow_version': tf.__version__,
}
with open(OUT_DIR / 'cnn_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps({k: meta[k] for k in ('test_metrics', 'train_metrics_clean', 'val_metrics',
                                       'overfitting_gap_train_minus_test_accuracy', 'temperature')}, indent=1))

In [ ]:
# Phase 10: Demo crops — one correctly classified test2019 crop per class (picked from the middle of
# the confidence range, not the easiest one), then clean the crop cache so the output stays small
test_conf = softmax(test_logits / TEMPERATURE).max(axis=1)
sample_dir = OUT_DIR / 'sample_crops'
sample_dir.mkdir(exist_ok=True)
sample_index = []
for label, sup in enumerate(supercategories):
    idx = [i for i in range(len(te_labels)) if te_labels[i] == label and test_pred[i] == label]
    if not idx:
        idx = [i for i in range(len(te_labels)) if te_labels[i] == label]
    if not idx:
        continue
    idx.sort(key=lambda i: test_conf[i])
    pick = idx[len(idx) // 2]
    shutil.copy(te_paths[pick], sample_dir / f'{sup}.jpg')
    sample_index.append({'supercategory': sup, 'label': label, 'file': f'{sup}.jpg',
                         'test_confidence': round(float(test_conf[pick]), 3)})
with open(sample_dir / 'index.json', 'w') as f:
    json.dump(sample_index, f, indent=2)

shutil.rmtree(CROP_DIR, ignore_errors=True)
print('Exported:', sorted(p.name for p in OUT_DIR.iterdir()))